In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
import os
import qnas_config as cfg
from util import check_files, load_yaml
from cnn.input import GenericDataLoader
from cnn import model, model_resnet
import torch
from sklearn.metrics import confusion_matrix
import torch.nn as nn

from medmnist import INFO, Evaluator
from torch.profiler import profile, record_function, ProfilerActivity


In [3]:
phase = 'retrain'
experiment_path = os.path.join("experiments_pathmnist", "resnet_18")
config_file = 'config_files_med/config2.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'retrain_folder': 'retrai_F10_1',
    'data_path': 'pathmnist_data',
    'dataset': 'pathmnist',
    'log_level': 'INFO',
    'max_epochs': 300,
    'epochs_to_eval': 10,
    'batch_size': 128,
    'eval_batch_size': 128,
    'limit_data': False,
    'num_workers': 4,
    'device': 'cuda:0',
    'lr_scheduler': 'None',
    'data_augmentation': False,
    'model_flag': 'resnet18',
}
args['input_shape'] = [128, 3, 28, 28]

In [4]:
dataset_info = load_yaml(os.path.join(args['data_path'], 'data_info.txt'))
args['num_classes'] = dataset_info["num_classes"]

In [5]:
# check_files(args['experiment_path'])
# config = cfg.ConfigParameters(args, phase=phase)
# config.get_parameters()

# fn_dict=config.fn_dict

In [6]:
# config.load_evolved_data(experiment_path=experiment_path)
# params = config.train_spec

In [7]:
#evolved_params = config.evolved_params

In [8]:
# params['net_list'] = evolved_params['net']
# params['fn_dict'] = fn_dict
# params['num_classes'] = 9
# params['input_shape'] = [128,3, 28, 28]


In [9]:
def reset_and_load_best_model(params, best_model_path, device):
    # Reinitialize the original model
    
    best_model = model.NetworkGraph(num_classes=params["num_classes"], mu=0.99)
    filtered_dict = {key: item for key, item in params['fn_dict'].items() if key in params['net_list']}
    best_model.create_functions(fn_dict=filtered_dict, net_list=params['net_list'])

    input_random = torch.randn(params['input_shape'])
    _ = best_model(input_random)
    # Load the state dictionary of the best model into the new model
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.to(device)

    return best_model

In [10]:
def reset_and_load_best_model_resnet(params, best_model_path):
    # Reinitialize the original model
    
    # Determine the model class based on the model_flag
    model_classes = {'resnet18': model_resnet.ResNet18, 'resnet50': model_resnet.ResNet50}
    if params['model_flag'] not in model_classes:
        raise ValueError(f"Unsupported model_flag: {params['model_flag']}")

    # Instantiate the model class
    best_model_class = model_classes[params['model_flag']]
    best_model = best_model_class(in_channels=params['input_shape'][1], num_classes=params['num_classes'])

    # Load the state dictionary of the best model into the new model
    best_model.load_state_dict(torch.load(best_model_path))
    best_model.to(params['device'])

    return best_model

In [11]:
def compute_metrics(model, data_loader, params):
    model.eval()
    all_labels = []
    all_predictions = []
    auc, acc = 0, 0
    y_score = torch.tensor([]).to(params['device'])

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(params['device']), labels.to(params['device'])
            y_logits = model(inputs)
            _, predicted = y_logits.max(1)
            output = y_logits.softmax(dim=-1)
            y_score = torch.cat((y_score, output), 0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
        
        if params['dataset'] != 'cifar10':
            y_score = y_score.cpu().detach().numpy()
            evaluator = Evaluator(params['dataset'], split='test', root=params['data_path'])
            metrics = evaluator.evaluate(y_score)
            auc, acc = metrics

    conf_matrix = confusion_matrix(all_labels, all_predictions)
    return conf_matrix, auc, acc

In [12]:
def evaluate(model, criterion, data_loader, params, test=True):
    model.eval()
    eval_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(params['device']), labels.to(params['device'])
            y_logits = model(inputs)
            
            labels = labels.squeeze().long() # medmnist
            loss = criterion(y_logits, labels)
            eval_loss += loss.item()
            _, predicted = y_logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total
    eval_loss /= len(data_loader)
    
    if test:
        confusion_matrix, auc, acc = compute_metrics(model, data_loader, params)
        return eval_loss, accuracy, auc, acc , confusion_matrix
    return eval_loss, accuracy

In [25]:
data_loader = GenericDataLoader(params=args)

In [26]:
#best_model = reset_and_load_best_model(params, os.path.join(experiment_path, 'retrain_2', 'best_model.pth'), args['device'])
best_model = reset_and_load_best_model_resnet(args, os.path.join(experiment_path, 'retrain_F10_1', 'best_model.pth'))
criterion = nn.CrossEntropyLoss()

In [27]:
test_loader = data_loader.get_loader(for_train=False, pin_memory_device=args['device'])

In [28]:
evaluated = evaluate(best_model, criterion, test_loader, args, test=True)
evaluated

(0.3179818989316884,
 93.27298050139275,
 0.9940784751333156,
 0.9327298050139275,
 array([[1285,    0,    0,    0,    2,   49,    0,    0,    2],
        [   0,  847,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,  313,    0,    1,   19,    0,    5,    1],
        [   0,    0,    0,  633,    0,    0,    0,    0,    1],
        [  37,   13,    1,    0,  940,    4,   18,    6,   16],
        [   0,   17,   21,    0,    1,  528,    0,   25,    0],
        [   0,    0,    0,    0,    1,    1,  719,    0,   20],
        [   0,    0,   36,    0,    5,   82,    3,  244,   51],
        [   0,    0,    3,   22,    0,    1,   17,    2, 1188]]))

In [29]:
# Measure inference time
inference_images = next(iter(test_loader))[0][:10].to(args['device'])

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],profile_memory=True, record_shapes=True) as prof:
    with record_function("model_inference"):
        best_model(inference_images)

model_memory_usage = sum(event.cuda_memory_usage for event in prof.key_averages()) / (1024 ** 2)
cpu_inference_time = prof.key_averages()[0].cpu_time
cuda_inference_time = prof.key_averages()[0].cuda_time
total_params = sum(p.numel() for p in best_model.parameters())
total_trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)

STAGE:2024-03-25 18:23:09 173548:173548 ActivityProfilerController.cpp:311] Completed Stage: Warm Up
STAGE:2024-03-25 18:23:09 173548:173548 ActivityProfilerController.cpp:317] Completed Stage: Collection
STAGE:2024-03-25 18:23:09 173548:173548 ActivityProfilerController.cpp:321] Completed Stage: Post Processing


In [30]:
cpu_inference_time, cuda_inference_time, total_params

(12350.0, 1366.0, 11173449)